# OSIXBAY Genesis — ECO-LONG-13 CPU batch
Run a **non-canonical experimental ecology audit** using the prepackaged private-source snapshot. Upload the ZIP generated by the GitHub workflow as a Kaggle Dataset, attach it under **Input**, use **Accelerator: None**, **Internet: Off**, then run this cell. All results stay in `/kaggle/working/eco_long13_results/`. Nothing deploys or merges.


In [ ]:
from pathlib import Path
import os, sys, subprocess, zipfile

# Upload OSIXBAY_ECO_LONG13_KAGGLE_SOURCE.zip as a Kaggle Dataset first.
# In Kaggle notebook settings: Accelerator = None, Internet = Off.
bundles = list(Path("/kaggle/input").rglob("OSIXBAY_ECO_LONG13_KAGGLE_SOURCE.zip"))
assert len(bundles) == 1, (
    "Expected one OSIXBAY_ECO_LONG13_KAGGLE_SOURCE.zip in Kaggle Inputs. "
    "Upload the GitHub Actions artifact as a Kaggle Dataset."
)
working = Path("/kaggle/working/osixbay_eco_long13")
working.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(bundles[0]) as zf:
    if any(name.startswith("/") or ".." in Path(name).parts for name in zf.namelist()):
        raise RuntimeError("Unsafe archive member")
    zf.extractall(working)
assert (working / "gple_f01" / "eco_long13_kaggle_batch.py").is_file()

# First run: 3 seeds x 6 fractions = 18 runs, 1,200 abstract ticks each.
# For additional seeds, edit this list. Stay below 50 unique seeds per batch.
seeds = [11, 23, 37]
ticks = 1200
assert len(seeds) <= 50
args = [sys.executable, str(working / "gple_f01" / "eco_long13_kaggle_batch.py"),
        "--seeds", ",".join(map(str, seeds)), "--ticks", str(ticks),
        "--outdir", "/kaggle/working/eco_long13_results"]
env = os.environ.copy()
env["PYTHONPATH"] = str(working / "gple_f01")
subprocess.run(args, check=True, cwd=working, env=env, timeout=1800)

import json
summary_path = Path("/kaggle/working/eco_long13_results/summary.json")
summary = json.loads(summary_path.read_text())
print("Completed", len(summary["cases"]), "cases")
for fraction in summary["fractions"]:
    matches = [x for x in summary["cases"] if x["fraction"] == fraction]
    print(f"Fraction {fraction:.0%}: births={sum(x['births'] for x in matches)}, "
          f"juvenile deaths={sum(x['juvenile_deaths'] for x in matches)}, "
          f"final survivors={sum(x['final_plants'] for x in matches)}")
print("Results: /kaggle/working/eco_long13_results/")
